In [ ]:
import os

# Replace the path below with your actual git.exe path
os.environ['GIT_PYTHON_GIT_EXECUTABLE'] = r'C:\Program Files\Git\cmd\git.exe'
os.environ['GIT_PYTHON_REFRESH'] = 'quiet'

import git
print(git.Git().version())


In [2]:
# Step 1: Import required libraries
import pandas as pd
import string

# Load your datasets
places = pd.read_csv("places.csv")
reviews = pd.read_csv("reviews.csv")

In [ ]:
places.head()

In [ ]:
reviews.head()

## Data Preprocessing

Group reviews

In [ ]:
# Group reviews to summarize per place_id
reviews_grouped = (
    reviews
    .groupby("place_id", as_index=False)
    .agg({
        "review_id": "count",            # how many reviews per shop
        "text": lambda x: list(x)        # optional: keep list of review texts
    })
    .rename(columns={"review_id": "review_count"})
)
reviews_grouped.head()


In [ ]:
# Merge grouped reviews with places, now also including 'category'
shops_r = reviews_grouped.merge(
    places[["place_id", "name", "address", "lat", "lng", "user_ratings_total", "category"]],
    on="place_id",
    how="inner",
    validate="1:1"
)
shops_r.head()

### Quick overview of the data

In [ ]:
print("Places dataset-")
print(f"Rows: {places.shape[0]}, Columns: {places.shape[1]}")

print("Reviews dataset-")
print(f"Rows: {reviews.shape[0]}, Columns: {reviews.shape[1]}")

print("Grouped reviews dataset-")
print(f"Rows: {reviews_grouped.shape[0]}, Columns: {reviews_grouped.shape[1]}")

print("Merged dataset-")
print(f"Rows: {shops_r.shape[0]}, Columns: {shops_r.shape[1]}")

**Check for missingness**

In [ ]:
# Count missing values in each column
shops_r.isna().sum()


**Check for duplicates**

In [ ]:
# Verify one unique row per place
duplicate_places = shops_r["place_id"].duplicated().sum()
print(f"Duplicate place_ids: {duplicate_places}")


In [ ]:
#print("Max reviews seen:", shops_r["review_count"].max())
#assert shops_r["review_count"].max() <= 5, "Unexpected: a place has >5 reviews"


In [ ]:
# Find places with >5 reviews
shops_r[shops_r["review_count"] > 5][["place_id", "name", "review_count"]]


In [ ]:
# Look at the actual review texts for one
pid = shops_r.loc[shops_r["review_count"] > 5, "place_id"].iloc[0]
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == pid, "text"].values[0], start=1):
    print(f"{idx}. {review}\n")


### Text Cleaning and Preprocessing

In [ ]:
import re

def clean_review_text(text):
    """Normalize review text for NLP."""
    text = str(text).lower()                          # lowercase
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"@\w+", " ", text)                  # remove mentions
    text = re.sub(r"#\w+", " ", text)                  # remove hashtags
    text = re.sub(r"[^\w\s]", " ", text)               # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()           # normalize spaces
    return text

# Create a new column with cleaned reviews for each shop
shops_r["clean_texts"] = shops_r["text"].apply(lambda reviews: [clean_review_text(r) for r in reviews])
shops_r.head()


In [ ]:
shops_r.head()

### Tokenisation

In [ ]:
import re

# A basic stopword list (can be expanded later)
STOPWORDS = set("""
a about above after again against all am an and any are as at be because been before being below
between both but by can did do does doing down during each few for from further had has have having
he her here hers herself him himself his how i if in into is it its itself just me more most my
myself no nor not of off on once only or other our ours ourselves out over own same she should so
some such than that the their theirs them themselves then there these they this those through to too
under until up very was we were what when where which while who whom why will with you your yours
yourself yourselves
""".split())

def tokenize_and_remove_stopwords(text):
    """Split text into tokens and remove common stopwords."""
    tokens = re.findall(r"[a-z']+", text)  # words only
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens


Clean empty reviews

In [ ]:
# Remove shops where any review text in the list is empty or NaN
shops_r["clean_texts"] = shops_r["clean_texts"].apply(
    lambda reviews: [r for r in reviews if r.strip() not in ("", "nan")]
)

# Drop rows where the resulting list is empty (no valid reviews left)
shops_r = shops_r[shops_r["clean_texts"].apply(len) > 0].copy()


In [ ]:
# Apply to each review in every shop
shops_r["tokens"] = shops_r["clean_texts"].apply(
    lambda reviews: [tokenize_and_remove_stopwords(r) for r in reviews]
)

shops_r.head(1)


In [ ]:
# Look at tokens for a random shop
import random
sample_tokens = random.choice(shops_r["tokens"].values)
for i, review_tokens in enumerate(sample_tokens, start=1):
    print(f"Review {i}: {review_tokens}")


Drop single character tokens

In [ ]:
import re

def clean_tokens(tokens):
    out = []
    for t in tokens:
        if t == "s":               # drop possessive leftovers
            continue
        if len(t) < 2:             # drop 1-char tokens
            continue
        if re.fullmatch(r"\d+", t):# drop pure numbers
            continue
        out.append(t)
    return out

# apply to each review’s token list
shops_r["tokens_clean"] = shops_r["tokens"].apply(lambda reviews: [clean_tokens(toks) for toks in reviews])
shops_r = shops_r.drop('tokens', axis=1)
shops_r.head(1)


**Categorizing the places and reviews**

In [ ]:
# Create four different DataFrames by category
rest_df= shops_r[shops_r["category"]== "restaurant"]
park_df= shops_r[shops_r["category"]== "park"]
mall_df= shops_r[shops_r["category"]== "shopping mall"]
tour_df= shops_r[shops_r["category"]== "tourist attraction"]

test_df=rest_df.copy()


In [ ]:
# Show counts for each category
print("Restaurants:", len(rest_df))
print("Parks:", len(park_df))
print("Shopping Malls:", len(mall_df))
print("Tourist Attractions:", len(tour_df))

In [ ]:
# Restaurants
rest_df.head()

In [ ]:
# Parks
park_df.head()

In [ ]:
# Malls
mall_df.head()

In [ ]:
# Tourist Attractions
tour_df.head()

### Keyword extraction

In [ ]:
# Define keywords
KEYWORDS = {
    "restaurant": {"clean", "cozy", "spacious", "crowded", "busy", "quiet", "noisy", "friendly", "rude",
        "welcoming", "attentive", "professional", "efficient", "slow", "quick", "prompt", "helpful",
        "approachable", "hygienic", "dirty", "tidy", "spotless", "comfortable", "uncomfortable", "overpriced",
        "expensive", "cheap", "affordable", "reasonable", "value", "worthwhile", "overrated", "excellent",
        "great", "amazing", "lovely", "fantastic", "outstanding", "average", "decent", "disappointing",
        "terrible", "awful", "poor", "superb", "nice", "pleasant", "beautiful", "gorgeous", "atmospheric",
        "decorated", "modern", "traditional", "inviting", "stylish", "safe", "unsafe"},
    "park": {"green", "lush", "clean", "peaceful", "quiet", "serene", "tranquil", "calm", "relaxing",
        "spacious", "open", "crowded", "busy", "safe", "unsafe", "family-friendly", "child-friendly",
        "pet-friendly", "dog-friendly", "accessible", "inclusive", "welcoming", "tidy", "hygienic", "dirty",
        "polluted", "beautiful", "scenic", "picturesque", "refreshing", "natural", "breezy", "shady",
        "sunny", "greenery", "landscaped", "maintained", "unkept", "secure", "unsafe", "peaceful",
        "quiet", "relaxing", "recreational", "sporty", "vibrant", "energetic", "playful", "safe",
        "clean", "refreshing", "lovely"},
    "shopping mall": {"spacious", "crowded", "busy", "quiet", "safe", "secure", "unsafe", "modern", "stylish",
        "clean", "tidy", "dirty", "hygienic", "accessible", "inclusive", "welcoming", "organized", "chaotic",
        "bright", "well-lit", "dark", "confusing", "easy", "navigable", "sprawling", "compact", "big",
        "huge", "small", "cramped", "air-conditioned", "comfortable", "uncomfortable", "overpriced", "expensive",
        "affordable", "cheap", "reasonable", "family-friendly", "child-friendly", "crowded", "popular", "busy",
        "trendy", "upscale", "luxury", "basic", "ordinary", "modernized", "outdated", "stylish", "inviting",
        "safe", "clean", "friendly"},
    "tourist attraction": {"historic", "ancient", "modern", "beautiful", "gorgeous", "scenic", "picturesque", "breathtaking",
        "majestic", "grand", "iconic", "famous", "popular", "crowded", "busy", "peaceful", "quiet", "serene",
        "clean", "tidy", "dirty", "unsafe", "safe", "secure", "accessible", "welcoming", "inclusive", "touristy",
        "authentic", "cultural", "traditional", "vibrant", "colorful", "energetic", "spiritual", "sacred",
        "artistic", "creative", "inspiring", "memorable", "remarkable", "unique", "extraordinary", "ordinary",
        "overrated", "expensive", "affordable", "reasonable", "educational", "informative", "guided", "interactive",
        "family-friendly", "child-friendly", "adventurous", "photogenic"}
}

In [ ]:
def add_keywords(df, category):
    vocab = KEYWORDS[category]
    df = df.copy()
    df["keywords"] = df["tokens_clean"].apply(
        lambda reviews: sorted({w for toks in reviews for w in toks if w in vocab})
    )
    return df


In [ ]:
rest_df = add_keywords(rest_df, "restaurant")
park_df = add_keywords(park_df, "park")
mall_df = add_keywords(mall_df, "shopping mall")
tour_df = add_keywords(tour_df, "tourist attraction")

test_df = add_keywords(test_df, "restaurant")

In [ ]:
test_df[["place_id","name","keywords"]].head()

## Sentiment analysis

In [ ]:
!pip install vaderSentiment


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# initialize analyzer
analyzer = SentimentIntensityAnalyzer()


In [ ]:
def score_keywords(keywords):
    if isinstance(keywords, str):
        keywords = [keywords]
    elif not isinstance(keywords, list):
        return []

    return [analyzer.polarity_scores(kw)["compound"] for kw in keywords if isinstance(kw, str)]


In [ ]:
test_df["kw_scores"] = test_df["keywords"].apply(score_keywords)


In [ ]:
# peek one row
test_df[["place_id","name","kw_scores"]].head(1)

In [ ]:
test_df["keywords"].iloc[0]  # This should be a list of lists, not a long string

In [ ]:
# Save to CSV
test_df.to_csv("test_df_with_keyword_scores.csv", index=False)


In [ ]:
def aggregate_sentiment(kw_scores):
    scores = []
    for score in kw_scores:
        if isinstance(score, list):
            scores.extend(score)
        elif isinstance(score, (int, float)):
            scores.append(score)

    if not scores:
        return pd.Series([0.0, 0.0, 0.0, "neutral"], 
                         index=["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"])

    avg = sum(scores) / len(scores)
    pos = sum(1 for s in scores if s > 0.05) / len(scores)
    neg = sum(1 for s in scores if s < -0.05) / len(scores)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series([avg, pos, neg, label],
                     index=["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"])


In [ ]:
test_df[["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]] = (
    test_df["kw_scores"].apply(aggregate_sentiment)
)

# Repeat similarly for park_df, mall_df, tour_df if needed


In [ ]:
test_df[["place_id","name","kw_scores","avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]].head(1)

#### SAMPLE TESTING

In [ ]:
import random

random.seed(42)
# Randomly select 30 unique place_ids
sample_ids = random.sample(list(test_df["place_id"].dropna().unique()), 30)
sample_df = test_df[test_df["place_id"].isin(sample_ids)]

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)


#### TEXTBLOB

In [ ]:
from textblob import TextBlob

In [ ]:
def get_textblob_sentiment(texts):
    if not isinstance(texts, list):
        return []
    return [TextBlob(review).sentiment.polarity for review in texts]


In [ ]:
sample_df["tb_scores"] = sample_df["clean_texts"].apply(get_textblob_sentiment)

In [ ]:
def aggregate_textblob_sentiment(tb_scores):
    if not isinstance(tb_scores, list) or len(tb_scores) == 0:
        return pd.Series([0.0, 0.0, 0.0, "neutral"], 
                         index=["tb_avg", "tb_pos", "tb_neg", "tb_label"])

    tb_avg = sum(tb_scores) / len(tb_scores)
    pos = sum(1 for s in tb_scores if s >= 0.05) / len(tb_scores)
    neg = sum(1 for s in tb_scores if s <= -0.05) / len(tb_scores)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series([tb_avg, pos, neg, label], 
                     index=["tb_avg", "tb_pos", "tb_neg", "tb_label"])


In [ ]:
sample_df[["tb_avg", "tb_pos", "tb_neg", "tb_label"]] = (
    sample_df["tb_scores"].apply(aggregate_textblob_sentiment)
)

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

#### BERT

In [ ]:
pip install transformers torch


In [ ]:
from transformers import pipeline

# Load BERT sentiment analysis pipeline
bert_classifier = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")


In [ ]:
# Define function to get sentiment per review
def get_bert_sentiment_per_review(cleaned_reviews):
    if not isinstance(cleaned_reviews, list):
        return []

    results = []
    for review in cleaned_reviews:
        if isinstance(review, str) and review.strip():
            pred = bert_classifier(review[:512])[0]  # Truncate to 512 tokens
            results.append((pred["label"].lower()))
    return results

In [ ]:
sample_df["bert_sentiment"] = sample_df["clean_texts"].apply(get_bert_sentiment_per_review)


In [ ]:
sample_df.head(1)

In [ ]:
# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

In [ ]:
def aggregate_bert(bert_labels_per_place):
    total = len(bert_labels_per_place)
    if total == 0:
        return pd.Series([0.0, 0.0, "neutral"],
                         index=["bert_pos", "bert_neg", "bert_label"])
    
    pos = sum(1 for label in bert_labels_per_place if label == "positive")
    neg = sum(1 for label in bert_labels_per_place if label == "negative")
    
    pct_pos = pos / total
    pct_neg = neg / total
    
    if pos > neg:
        overall = "positive"
    elif neg > pos:
        overall = "negative"
    else:
        overall = "neutral"
    
    return pd.Series([pct_pos, pct_neg, overall],
                     index=["bert_pos", "bert_neg", "bert_label"])


In [ ]:
sample_df[["bert_pos", "bert_neg", "bert_label"]] = sample_df["bert_sentiment"].apply(aggregate_bert)

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

#### MODEL COMPARISON

In [ ]:
# Load the file
comp_df = pd.read_excel("manual labels for reviews.xlsx")

In [ ]:
man_agg = []
for i in range(len(comp_df)):
    labels = comp_df.loc[i, "manual_labels"].split(",")
    labels = [l.strip().lower() for l in labels]  # normalize case just in case
    counts = pd.Series(labels).value_counts()

    if (counts == counts.max()).sum() > 1:
        agg = "neutral"
    else:
        agg = counts.idxmax()

    man_agg.append(agg)

comp_df["man_agg"] = man_agg


In [ ]:
comp_df.head(1)

In [ ]:
from sklearn.metrics import accuracy_score

# Compare manual labels vs each model
vader_acc = accuracy_score(comp_df["man_agg"], comp_df["sentiment_label"])
textblob_acc = accuracy_score(comp_df["man_agg"], comp_df["tb_label"])
bert_acc = accuracy_score(comp_df["man_agg"], comp_df["bert_label"])

# Print results
print("Accuracy vs Manual Labels:")
print(f"VADER Accuracy     : {vader_acc:.2f}")
print(f"TextBlob Accuracy  : {textblob_acc:.2f}")
print(f"BERT Accuracy      : {bert_acc:.2f}")


In [ ]:
comp_df['man_agg'].describe()

**Data is imbalanced. Positive counts are 24/30, so accuracy alone is misleading. There classification report can provide deeper insight on model selection**

In [ ]:
from sklearn.metrics import classification_report

print("VADER:")
print(classification_report(comp_df["man_agg"], comp_df["sentiment_label"]))

print("TextBlob:")
print(classification_report(comp_df["man_agg"], comp_df["tb_label"]))

print("BERT:")
print(classification_report(comp_df["man_agg"], comp_df["bert_label"]))


#### Classification Reports

**VADER**:
- **Strengths**: High accuracy on positive sentiment (Precision = 0.89, Recall = 1.00)
- **Weaknesses**: Very poor recall on negative reviews (Recall = 0.20), meaning it often misses them.
- **Neutral detection**: Correctly detected the only neutral sample.

**TextBlob**:
- **Strengths**: High precision and recall across all categories. Neutral and positive classes were perfectly predicted.
- **Weaknesses**: Slight underperformance on negative recall (0.80), but overall very strong.

**BERT**:
- **Strengths**: Most balanced performance, excellent at detecting both positive and negative reviews.
- **Weaknesses**: Missed just one positive review (Recall = 0.96).
- **Neutral detection**: Perfect.

---

#### Interpretation

- **Accuracy alone is misleading** due to class imbalance (80% positive).
- **VADER** struggles with subtle or implied negativity.
- **TextBlob** offers excellent performance with simple implementation.
- **BERT** is the most robust and balanced model — ideal for production-level sentiment tasks.

---

#### ✅ Final Model Selection: BERT

Based on the evaluation of sentiment models (VADER, TextBlob, and BERT) against manual labels:

- **BERT** provided the **most balanced and accurate** performance across all sentiment classes.
- It achieved:
  - **100% recall for negative and neutral** sentiments
  - **96% recall for positive** sentiments
  - **Overall accuracy of 97%**, matching TextBlob but with **stronger performance on the negative class**.

---

In [ ]:
rest_df["bert_sentiment"] = rest_df["clean_texts"].apply(get_bert_sentiment_per_review)


In [ ]:
rest_df.head(1)

In [ ]:
rest_df[["bert_pos", "bert_neg", "bert_label"]] = rest_df["bert_sentiment"].apply(aggregate_bert)

# Save to Excel
rest_df.to_excel("restaurants_sentiment.xlsx", index=False)

In [ ]:
park_df["bert_sentiment"] = park_df["clean_texts"].apply(get_bert_sentiment_per_review)
park_df.head(1)

In [ ]:
park_df[["bert_pos", "bert_neg", "bert_label"]] = park_df["bert_sentiment"].apply(aggregate_bert)

# Save to Excel
park_df.to_excel("parks_sentiment.xlsx", index=False)

In [ ]:
mall_df["bert_sentiment"] = mall_df["clean_texts"].apply(get_bert_sentiment_per_review)
mall_df.head(1)

In [ ]:
mall_df[["bert_pos", "bert_neg", "bert_label"]] = mall_df["bert_sentiment"].apply(aggregate_bert)

# Save to Excel
mall_df.to_excel("malls_sentiment.xlsx", index=False)

In [ ]:
tour_df["bert_sentiment"] = tour_df["clean_texts"].apply(get_bert_sentiment_per_review)
tour_df.head(1)

In [ ]:
tour_df[["bert_pos", "bert_neg", "bert_label"]] = tour_df["bert_sentiment"].apply(aggregate_bert)

# Save to Excel
tour_df.to_excel("tourist_sentiment.xlsx", index=False)

### Aspect-Based Sentiment Analysis (ABSA)

In [11]:
# Install required libraries
!pip install transformers pandas openpyxl hf_xet --quiet 


In [12]:
from transformers import pipeline

# Use a small and public model (multilingual, fast)
sentiment_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")


Device set to use cpu


In [14]:
import ast

# Load your file
rest_aspect = pd.read_excel("restaurants_sentiment.xlsx")
# Convert stringified list to actual list
rest_aspect["clean_texts"] = rest_aspect["clean_texts"].apply(ast.literal_eval)

In [ ]:
# Define mapping from model labels to numeric values
label_map = {
    "1 star": 1,
    "2 stars": 2,
    "3 stars": 3,
    "4 stars": 4,
    "5 stars": 5
}

# Function to score list of reviews
def score_reviews(review_list):
    results = sentiment_model(review_list)
    scores = [label_map[r['label']] for r in results]
    return {
        "avg_sentiment": sum(scores)/len(scores),
        "max_sentiment": max(scores),
        "min_sentiment": min(scores),
        "positive_pct": sum(1 for s in scores if s >= 4) / len(scores),
        "review_count": len(scores),
        "individual_scores": scores
    }
